# GNN Training — PSR Pipeline (Colab T4)

**Issue #40**: Integrate + Benchmark GNN Training Pipeline

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `physiomio.zip` (train.pt / val.pt / test.pt) to your Google Drive
3. Update `DRIVE_DATASET_PATH` below to point to where you put the files
4. Set `REPO_URL` and `GITHUB_TOKEN` for your psr-pipeline repo


In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
REPO_URL        = "https://github.com/post-stroke-rehab/psr-pipeline.git"
GITHUB_TOKEN    = ""   # paste your PAT here if the repo is private

# Path inside your Google Drive where you unzipped physiomio.zip
# e.g. if you put train.pt at MyDrive/physiomio/train.pt → set to "/content/drive/MyDrive/physiomio"
DRIVE_DATASET_PATH = "/content/drive/MyDrive/physiomio"

# Training hyperparameters
EPOCHS     = 200
BATCH_SIZE = 128
LR         = 1e-3
SEED       = 42
# ──────────────────────────────────────────────────────────────────────────────

## 1. Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found — change runtime type to T4!"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Clone repo

In [ ]:
import os

if GITHUB_TOKEN:
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
else:
    clone_url = REPO_URL

if not os.path.exists("/content/psr-pipeline"):
    !git clone {clone_url} /content/psr-pipeline
else:
    print("Repo already cloned — pulling latest changes")
    !cd /content/psr-pipeline && git pull

%cd /content/psr-pipeline
!git log --oneline -5

## 3. Install dependencies

In [ ]:
!pip install -q torch_geometric
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)").html
!pip install -q PyWavelets optuna scikit-learn matplotlib pyarrow pandas

## 4. Mount Google Drive and copy dataset

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import shutil, os

dest = "/content/psr-pipeline/datasets/processed/physiomio"
os.makedirs(dest, exist_ok=True)

for split in ["train.pt", "val.pt", "test.pt"]:
    src = os.path.join(DRIVE_DATASET_PATH, split)
    dst = os.path.join(dest, split)
    if not os.path.exists(dst):
        print(f"Copying {split}...")
        shutil.copy2(src, dst)
    else:
        print(f"{split} already present — skipping copy")

# Quick sanity check
import torch
for split in ["train.pt", "val.pt", "test.pt"]:
    d = torch.load(os.path.join(dest, split), map_location="cpu", weights_only=True)
    print(f"{split}: X={tuple(d['X'].shape)}, y={tuple(d['y'].shape)}")

## 5. Run GNN training

In [ ]:
os.chdir("/content/psr-pipeline")

!python training/train.py \
    --model gnn \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --seed {SEED} \
    --results_dir results

## 6. Inspect results

In [ ]:
import json

with open("results/gnn/metrics.json") as f:
    metrics = json.load(f)

print("=== Test Metrics ===")
for k, v in metrics.items():
    if k != "per_finger":
        print(f"  {k}: {v:.4f}")

print("\n=== Per-Finger ===")
for finger, stats in metrics["per_finger"].items():
    print(f"  {finger}: F1={stats['f1']:.4f}, AUROC={stats['auroc']:.4f}, AUPRC={stats['auprc']:.4f}")

In [ ]:
from IPython.display import Image, display, Markdown
import matplotlib.pyplot as plt

for img in ["loss_curves.png", "roc_curve.png", "pr_curve.png", "confusion_matrices.png"]:
    path = f"results/gnn/{img}"
    if os.path.exists(path):
        print(f"--- {img} ---")
        display(Image(path))

In [ ]:
# Print the summary.md
with open("results/gnn/summary.md") as f:
    print(f.read())

## 7. Save results to Google Drive

In [ ]:
# Copy entire results/gnn/ folder back to Drive so you don't lose it when Colab disconnects
drive_results = "/content/drive/MyDrive/psr-results/gnn"
os.makedirs(drive_results, exist_ok=True)

import shutil
for fname in os.listdir("results/gnn"):
    src = os.path.join("results/gnn", fname)
    dst = os.path.join(drive_results, fname)
    shutil.copy2(src, dst)
    print(f"Saved: {fname}")

print(f"\nAll results saved to Google Drive: {drive_results}")